# Part I: Language Model Training and Comparison

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
import time
import math
import re


c:\Users\jenni\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("stanfordnlp/imdb")
train_lines = dataset["train"]["text"]

In [3]:
def tokenize(line):
    line = line.strip().lower()
    if not line:
        return []
    # Skip Wikipedia section headers like "== History =="
    if re.fullmatch(r"=+\s*.*\s*=+", line):
        return []
    # Extract words and punctuation as tokens
    return re.findall(r"\w+|[^\w\s]", line)

In [4]:
MAX_LINES = 300  # limit for speed (increase if you want a stronger model)

tokens = []
start_train1 = time.time()
for line in train_lines[:MAX_LINES]:
    toks = tokenize(line)
    if toks:
        tokens.extend(["<s>"] + toks + ["</s>"])

print("Total tokens:", len(tokens))

# Vocabulary = set of all unique tokens we observed
vocab = set(tokens)
print("Vocab size:", len(vocab))
end_train1 = time.time()

Total tokens: 86214
Vocab size: 8199


In [5]:
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

In [ ]:
encoded_text = [word_to_idx[word] for word in tokens]

In [7]:
data = torch.tensor(encoded_text, dtype = torch.long)

In [8]:
# RNN-based LM
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(RNNLanguageModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        x = self.embedding(x) # Embedding layer
        output, hidden = self.rnn(x, hidden) # RNN layer
        output = self.fc(output) # Fully connected layer
        return output, hidden

In [9]:
# Generate batches of data
def get_batch(data, seq_len, batch_size):
    for i in range(0, len(data) - seq_len, seq_len):
        x = data[i:i+seq_len]
        y = data[i+1:i+seq_len+1]
        yield x.view(-1, seq_len), y.view(-1, seq_len)

In [10]:
# Hyperparameters
vocab_size = len(vocab)
embed_size = 128
hidden_size = 256
num_layers = 2
seq_len = 100
batch_size = 32
num_epochs = 5
learning_rate = 0.001


In [11]:
# Model, loss, and optimizer
model = RNNLanguageModel(vocab_size, embed_size, hidden_size, num_layers)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [12]:
# Training loop with progress bar
start_train = time.time()
for epoch in range(num_epochs):
    hidden = None # Initialize hidden state
    total_loss = 0
    progress_bar = tqdm(get_batch(data, seq_len, batch_size), desc=f"Epoch{epoch+1}/{num_epochs}")
    
    for x_batch, y_batch in progress_bar:
        optimizer.zero_grad()
        output, hidden = model(x_batch, hidden)
        hidden = hidden.detach() # Detach hidden state for the next batch
        loss = criterion(output.view(-1, vocab_size), y_batch.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        # Update the progress bar with the current loss
        progress_bar.set_postfix(loss=loss.item())
    print(f"Epoch {epoch+1}/{num_epochs}, Total Loss: {total_loss:.4f}")

end_train = time.time()

Epoch1/5: 862it [00:32, 26.75it/s, loss=5.3] 


Epoch 1/5, Total Loss: 5142.4155


Epoch2/5: 862it [00:37, 23.11it/s, loss=4.79]


Epoch 2/5, Total Loss: 4378.9693


Epoch3/5: 862it [00:38, 22.49it/s, loss=4.35]


Epoch 3/5, Total Loss: 3924.6851


Epoch4/5: 862it [01:08, 12.50it/s, loss=4.05]


Epoch 4/5, Total Loss: 3547.8880


Epoch5/5: 862it [02:15,  6.37it/s, loss=3.82]

Epoch 5/5, Total Loss: 3234.3641


In [13]:
print("training time :", end_train - start_train)

training time : 312.10913014411926


In [28]:
print(loss)
print(total_loss)

tensor(3.8161, grad_fn=<NllLossBackward0>)
3234.364100217819


In [14]:
# Generate text
def generate_text(model, start_text, max_len=200):
    model.eval()
    # Convert starting text to indices and create input tensor
    input_ids = torch.tensor([word_to_idx[word] for word in start_text],
dtype=torch.long).unsqueeze(0) # Shape: (1, len(start_text))
    generated_text = list(start_text) # Store the generated words
    hidden = None # Initialize hidden state
    
    for _ in range(max_len):
        output, hidden = model(input_ids, hidden) # Forward pass
        next_word_id = output[:, -1, :].argmax(dim=-1).item() # Take the last step's output
        next_word = idx_to_word[next_word_id] # Convert to word
        generated_text.append(next_word)
        input_ids = torch.tensor([[next_word_id]], dtype=torch.long) # Update input with the predicted word
        if next_word == ".": # Stop generating at a full stop
            break
    return ' '.join(generated_text)


In [16]:
start_text = ["this","movie"]

start_inf = time.time()
print("Generated Text:", generate_text(model, start_text))
end_inf = time.time()

Generated Text: this movie is a crappy copy of a previously worthless island changed into something worthwhile .


In [18]:
print("inference time: ", end_inf - start_inf )

inference time:  0.030022859573364258
